In [1]:
from datetime import datetime

import asyncio
import time
import pandas as pd
import xlwings as xw

from ib_insync import IB, Option
from Class_IBKRClient import *
ibkr = IBKRClient()

from Class_xlWings import *
xls = xlWings()

In [2]:
async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

In [3]:
async def get_all_options_for_conid(symbol, conId, sleep_secs=0.2):
    
    try:
        chains = await ibkr.ib.reqSecDefOptParamsAsync(
            underlyingSymbol=symbol,
            futFopExchange='',
            underlyingSecType='STK',
            underlyingConId=conId
        )
    except Exception as e:
        print(f"[{symbol}] Chain request failed: {e}")
        return pd.DataFrame()

    if not chains:
        print(f"[{symbol}] No option chains")
        return pd.DataFrame()

    chain = chains[0]

    if not chain.expirations or not chain.strikes:
        print(f"[{symbol}] No expirations or strikes")
        return pd.DataFrame()

    contracts = [
        Option(
            symbol=symbol,
            lastTradeDateOrContractMonth=expiry,
            strike=strike,
            right=right,
            exchange='SMART',
            currency='USD'
        )
        for expiry in chain.expirations
        for strike in chain.strikes
        for right in ('C', 'P')
    ]

    if not contracts:
        return pd.DataFrame()

    # qualify contracts asynchronously
    qualified = []
    chunk_size = 200
    for i in range(0, len(contracts), chunk_size):
        try:
            qualified_chunk = await ibkr.ib.qualifyContractsAsync(*contracts[i:i+chunk_size])
            qualified += qualified_chunk
            await asyncio.sleep(sleep_secs)
        except Exception as e:
            print(f"[{symbol}] Qualification chunk failed: {e}")

    rows = []
    for c in qualified:
        rows.append({
            'underlying_symbol': symbol,
            'underlying_conId': conId,
            'option_conId': c.conId,
            'expiry': c.lastTradeDateOrContractMonth,
            'strike': c.strike,
            'right': c.right,
            'exchange': c.exchange,
            'multiplier': c.multiplier,
            'currency': c.currency,
            'tradingClass': c.tradingClass
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        sht = wb.sheets[symbol + " Options"]
        current_time = datetime.now() 
        sht.range("A1").value = current_time
        sht.range("A2").options(index=False).value = df

In [ ]:
await start_ibkr()

wb = xw.Book("2026 Crypto Products Database.xlsx")
sht = wb.sheets["Bitcoin"]

tbl = sht.tables["btc_static_data_table"]
tbl_range = tbl.range

static_df = tbl_range.options(pd.DataFrame, index=False).value

input_df = (
    static_df
    .loc[
        (static_df.my_prod_type == 'equity') &
        (static_df.platform_id == 'IBKR'),
        ['platform_symbol', 'ibkr_conId']
    ]
    .dropna()
)
input_df['ibkr_conId'] = input_df['ibkr_conId'].astype(int)    
print(input_df)

tasks = [
    get_all_options_for_conid(row.platform_symbol, int(row.ibkr_conId))
    for row in input_df.itertuples(index=False)
]

await asyncio.gather(*tasks)

for i in range(5):
    print('\n')

print('app completed')

for i in range(5):
    print('\n')

IBKR connected: True
  platform_symbol  ibkr_conId
0            ARKB   677037663
1            BITB   677037658
2            BRRR   677037676
3             BTC   741192224
4            BTCW   676783288
5            EZBC   676783282
6            FBTC   676783301
7            GBTC   349966059
8            HODL   677037670
9            IBIT   677037673
[BRRR] No option chains


Error 200, reqId 97: No security definition has been found for the request, contract: Option(symbol='ARKB', lastTradeDateOrContractMonth='20260417', strike=11.0, right='C', exchange='SMART', currency='USD')
Error 200, reqId 98: No security definition has been found for the request, contract: Option(symbol='ARKB', lastTradeDateOrContractMonth='20260417', strike=11.0, right='P', exchange='SMART', currency='USD')
Error 200, reqId 99: No security definition has been found for the request, contract: Option(symbol='ARKB', lastTradeDateOrContractMonth='20260417', strike=12.0, right='C', exchange='SMART', currency='USD')
Error 200, reqId 100: No security definition has been found for the request, contract: Option(symbol='ARKB', lastTradeDateOrContractMonth='20260417', strike=12.0, right='P', exchange='SMART', currency='USD')
Error 200, reqId 143: No security definition has been found for the request, contract: Option(symbol='ARKB', lastTradeDateOrContractMonth='20260417', strike=34.0, right='C